# BÀI GIẢI — Restaurant Tips Dataset

Dùng để **đối chiếu sau khi tự làm**, không nên xem trước khi làm bài `practice_tips_exercise.ipynb`.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
df_raw = pd.read_csv(url)

np.random.seed(42)
n = len(df_raw)
idx_bill = np.random.choice(df_raw.index, size=int(0.05*n), replace=False)
idx_day = np.random.choice(df_raw.index, size=int(0.05*n), replace=False)
df_raw.loc[idx_bill, "total_bill"] = np.nan
df_raw.loc[idx_day, "day"] = np.nan

df_raw.head()

## A.1. Missing Values

In [ ]:
print(df_raw.isnull().sum())

df = df_raw.copy()

df["day"] = df["day"].fillna(df["day"].mode()[0])
df["total_bill"] = df["total_bill"].fillna(df["total_bill"].median())

print(df.isnull().sum())

## A.2. Correct Data Format

In [ ]:
print(df.dtypes)

for col in ["sex", "smoker", "day", "time"]:
    df[col] = df[col].astype("category")

print(df.dtypes)

## A.3. Cột mới & Standardization

In [ ]:
df["tip_pct"] = df["tip"] / df["total_bill"]

smoker_map = {"No": 0, "Yes": 1}
df["smoker_flag"] = df["smoker"].map(smoker_map).astype(int)

df[["total_bill","tip","tip_pct","smoker","smoker_flag"]].head()

## A.4. Normalization

In [ ]:
df["total_bill_normalized"] = df["total_bill"] / df["total_bill"].max()
df[["total_bill","total_bill_normalized"]].head()

## A.5. Binning

In [1]:
bins = np.linspace(df["total_bill"].min(), df["total_bill"].max(), 4)
df["bill_level"] = pd.cut(df["total_bill"], bins=bins, labels=["Low","Medium","High"], include_lowest=True)

print(df["bill_level"].value_counts())

plt.hist(df["total_bill"], bins=3, edgecolor="black")
plt.xlabel("Total bill")
plt.ylabel("Count")
plt.title("Phan phoi Total Bill theo 3 nhom")
plt.show()

NameError: name 'np' is not defined

## A.6. Dummy Variable

In [ ]:
dummy_day = pd.get_dummies(df["day"], prefix="day", dtype=int)
df = pd.concat([df, dummy_day], axis=1)
df.head()

## B.1. Data size, columns, dtypes

In [ ]:
print(df.shape)
print(df.columns.tolist())
print(df.dtypes)

## B.2. Invalid values

In [ ]:
print(df[(df["total_bill"] <= 0) | (df["tip"] < 0)])
print("So dong trung lap:", df.duplicated().sum())

## B.3. Thống kê mô tả

In [ ]:
# Group 1 - Central Tendency
print("Mean total_bill:", df["total_bill"].mean())
print("Median total_bill:", df["total_bill"].median())
print("Mode total_bill:", df["total_bill"].mode()[0])
print("Mean tip_pct:", df["tip_pct"].mean())

# Group 2 - Dispersion
print("Std total_bill:", df["total_bill"].std())
print("Var total_bill:", df["total_bill"].var())
print("Range total_bill:", df["total_bill"].max() - df["total_bill"].min())
q1, q3 = df["total_bill"].quantile([0.25, 0.75])
print("IQR total_bill:", q3 - q1)

# Group 3 - Location & Shape
print(df["tip_pct"].describe())
print("Skewness tip_pct:", df["tip_pct"].skew())

## C.1. Outlier detection (IQR) + Boxplot

In [ ]:
Q1 = df["total_bill"].quantile(0.25)
Q3 = df["total_bill"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5*IQR
upper_bound = Q3 + 1.5*IQR

outliers = df[(df["total_bill"] < lower_bound) | (df["total_bill"] > upper_bound)]
print("So luong outlier:", len(outliers))
outliers[["total_bill"]]

In [ ]:
plt.boxplot(df["total_bill"])
plt.ylabel("Total bill")
plt.title("Boxplot Total Bill")
plt.show()

## C.2. Histogram phân phối tip_pct

In [2]:
plt.hist(df["tip_pct"], bins=15, edgecolor="black")
plt.xlabel("Tip percentage")
plt.ylabel("Count")
plt.title("Phan phoi Tip Percentage")
plt.show()

NameError: name 'plt' is not defined

## C.3. Boxplot total_bill theo day (matplotlib thuần)

In [ ]:
groups = [df.loc[df["day"] == d, "total_bill"].dropna() for d in df["day"].cat.categories]
plt.boxplot(groups, labels=df["day"].cat.categories)
plt.xlabel("Day")
plt.ylabel("Total bill")
plt.title("Total Bill theo tung ngay")
plt.show()

## C.4. Scatter & Correlation

In [ ]:
plt.scatter(df["total_bill"], df["tip"], alpha=0.6)
plt.xlabel("Total bill")
plt.ylabel("Tip")
plt.title("Total Bill vs Tip")
plt.show()

corr_matrix = df[["total_bill","tip","size","tip_pct"]].corr()
print(corr_matrix)

In [ ]:
im = plt.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(im)
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=45)
plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns)
for i in range(len(corr_matrix)):
    for j in range(len(corr_matrix)):
        plt.text(j, i, f"{corr_matrix.iloc[i,j]:.2f}", ha="center", va="center", color="black")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## C.5. Subplots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,4))

axes[0].hist(df["total_bill"], bins=15, edgecolor="black")
axes[0].set_title("Total bill distribution")
axes[0].set_xlabel("Total bill")
axes[0].set_ylabel("Count")

axes[1].hist(df["tip"], bins=15, edgecolor="black", color="orange")
axes[1].set_title("Tip distribution")
axes[1].set_xlabel("Tip")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

## PHẦN D — Câu hỏi phân tích

In [ ]:
# Cau 1: Smoker vs Non-smoker tip_pct
print(df.groupby("smoker")["tip_pct"].mean())

**Insight câu 1:** Nhìn kết quả `groupby`, so sánh 2 giá trị mean để kết luận nhóm nào tip % cao hơn — thực tế dữ liệu tips cho thấy chênh lệch khá nhỏ giữa 2 nhóm.

In [ ]:
# Cau 2: Dinner vs Lunch total_bill
print(df.groupby("time")["total_bill"].mean())

**Insight câu 2:** Dinner thường có total_bill trung bình cao hơn Lunch.

In [ ]:
# Cau 3: size vs tip_pct
print(df.groupby("size")["tip_pct"].mean())

**Insight câu 3:** Thường thì bàn quá đông người (size lớn) lại có tip % thấp hơn — trái ngược với suy đoán ban đầu.

In [ ]:
# Cau 4: day vs total revenue
print(df.groupby("day")["total_bill"].sum().sort_values(ascending=False))

**Insight câu 4:** Cuối tuần (Sat/Sun) thường có tổng doanh thu cao nhất do đông khách hơn.

In [ ]:
# Cau 5: correlation total_bill-tip vs total_bill-tip_pct
print("Corr(total_bill, tip):", df["total_bill"].corr(df["tip"]))
print("Corr(total_bill, tip_pct):", df["total_bill"].corr(df["tip_pct"]))

**Insight câu 5:** `total_bill` và `tip` tương quan dương khá mạnh (hóa đơn lớn → tip nhiều hơn về số tuyệt đối), nhưng tương quan với `tip_pct` yếu hơn nhiều hoặc thậm chí hơi âm — vì % tip không tăng tuyến tính theo hóa đơn.

**Câu 6 — Insight tổng hợp (ví dụ):**

1. Hóa đơn càng lớn thì tiền tip tuyệt đối càng nhiều, nhưng tỷ lệ % tip không nhất thiết tăng theo.
2. Bữa tối có hóa đơn trung bình cao hơn bữa trưa, có thể do gọi nhiều món/đồ uống hơn.
3. Bàn đông người có xu hướng tip % thấp hơn bàn ít người — có thể do chia hóa đơn hoặc thói quen tip theo đầu người giảm.
4. Cuối tuần là thời điểm đông khách và doanh thu cao nhất trong tuần.